# KITP Effect of Future Arctic Sea Ice Loss on the Greenland Ice Sheet 

In [ ]:
from functools import partial
from pathlib import Path
import json                                                                                                                                                                        
import dask
import xarray as xr
import pint_xarray
import matplotlib.pylab as plt
import matplotlib as mpl
import nc_time_axis
import numpy as np
import pandas as pd
from dask.diagnostics import ProgressBar
from pism_terra.processing import preprocess_netcdf as preprocess
import cartopy.crs as ccrs 
import warnings                                                                                                                                                    
warnings.filterwarnings("ignore", message="Increasing number of chunks")
from cycler import cycler
import cmweather
from cmap import Colormap
cm = Colormap('tol:bright').to_matplotlib()
cm_cycler = cycler(color=cm.colors) 
cm_precip = Colormap("crameri:navia").to_matplotlib()
cm_rdbu = Colormap("crameri:vik_r").to_matplotlib()

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
delta_coder = xr.coders.CFTimedeltaCoder()

## Ice-sheet wide scalars

In [ ]:
data_dir = "/Volumes/LHS800DATA"

baseline_opts =  {'short_hand': 'baseline', 'color': (0 , 0, 0), 'ls': "dashed", 'title': "Baseline"}

exps_opts = {
        'pdSST-futArcSICSIT_pdSST-pdSICSIT': {'short_hand': 'pdSST-futArcSICSIT_pdSST-pdSICSIT', 'color': (0.0660, 0.4430, 0.7450), 'ls': "solid", 'title': 'Arctic sea ice loss (AGCM)'},
        'pa-futArcSIC-ext_pa-pdSIC-ext': {'short_hand': 'pa-futArcSIC-ext_pa-pdSIC-ext', 'color': (0.8660, 0.3290, 0), 'ls': "solid", "title": 'Arctic sea ice loss (AOGCM)'},
        'futSST-pdSIC_pdSST-pdSIC': {'short_hand': 'futSST-pdSIC_pdSST-pdSIC', 'color': (0.9290, 0.6940, 0.1250), 'ls': "solid", "title": "Global SST warming"},
        'pdSST-futArcSIC_pdSST-pdSIC': {'short_hand': 'pdSST-futArcSIC_pdSST-pdSIC', 'color': (0.5210, 0.0860, 0.8190), 'ls': "solid",'title': 'Arctic sea ice loss (AGCM + 2m ice)'},
        'futSST-futArcSIC-SUM_pdSST-pdSIC': {'short_hand': 'futSST-futArcSIC-SUM_pdSST-pdSIC', 'color': (0.2310, 0.6660, 0.1960), 'ls': "solid", "title": "Global SST warming + SIC loss (AGCM + 2m ice)"},
}

gt2mmsle = xr.DataArray(-1 / 361.8).pint.quantify("mm/Gt")

In [ ]:
pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

In [ ]:
%%time
baseline_file = Path(f"{data_dir}/2026_04_kitp_v4/output/scalar/scalar_g2400m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
baseline = xr.open_dataset(baseline_file,
                           chunks=None,
                           decode_times=time_coder,
                           decode_timedelta=delta_coder)
baseline = baseline.pint.quantify()

exp_files = [f for f in Path(f"{data_dir}/2026_04_kitp_v4/output/scalar/").glob("scalar_*2400m*.nc") if "HIRHAM5" not in f.name] 

def load_dataset(filename_or_obj: str | Path, **kwargs) -> xr.Dataset:
    with ProgressBar():
        ds = xr.open_mfdataset(filename_or_obj,
                               preprocess=partial(preprocess, **kwargs), 
                               engine="netcdf4",
                               parallel=True,
                               chunks=None,
                               data_vars="all",
                               join="outer",
                               decode_times=time_coder,
                               decode_timedelta=delta_coder)
        return ds
    
with dask.config.set(**{"array.slicing.split_large_chunks": False}):
    exps_ds = load_dataset(exp_files,
                          exp_regexp=r"id_.+?_((?:futSST|pdSST|pa)-\S+?)_\d{4}-\d{2}-\d{2}",
                          uq_regexp=None,   
                          uq_dim=None,
                          exp_dim="exp_id")
        
experiments = exps_ds.pint.quantify()



In [ ]:
pctls = [0.05, 0.5, 0.95]

baseline = baseline.expand_dims({"basin": ["GIS"]}) if ("basin" not in baseline.dims) else baseline

experiments = experiments.expand_dims({"basin": ["GIS"]}) if ("basin" not in experiments.dims) else experiments
obj_vars = [v for v in experiments.data_vars if experiments[v].dtype == object]                                                                                                                      
dss = []
for pctl in pctls:
    _ds = experiments.drop_vars(obj_vars).quantile(pctl, dim="gcm_id", skipna=False)
    _ds = _ds.expand_dims({"pctl": [pctl]})
    dss.append(_ds)
experiments_pctls = xr.concat(dss, dim="pctl")


## Plot ice sheet-wide scalars

In [ ]:

#config = json.loads(baseline.pism_config.item())
#res = f"{int(config["grid.dx"])}m"
res="2400m"
for basin_name in experiments_pctls.basin:
    basin = experiments_pctls.sel(basin=basin_name)
    basin_baseline = baseline.sel(basin=basin_name)
    with mpl.rc_context(rc=rc_params):
        fig, ax = plt.subplots(1, 1, sharex=True, figsize=(6.4, 3.2))
        _ds = basin_baseline
        ice_mass = _ds["ice_mass"].pint.to("Gt").pint.dequantify()
        ice_mass -= ice_mass.isel({"time": 0})
        slc = ice_mass * gt2mmsle
        slc.plot(ax=ax, color=baseline_opts["color"], ls=baseline_opts["ls"], label=baseline_opts["title"], lw=1)
        for exp_name, exp in exps_opts.items():
            _ds = basin.sel({"exp_id": exp_name})
            ice_mass = _ds["ice_mass"].pint.to("Gt").pint.dequantify()
            ice_mass -= ice_mass.isel({"time": 0})
            slc = ice_mass * gt2mmsle
            ax.fill_between(slc.sel({"pctl": 0.5}).time.values, slc.sel({"pctl": pctls[0]}), slc.sel({"pctl": pctls[-1]}),  
                                                                                              color=exp["color"], lw=0,
                                                                                             alpha=0.5)
            slc.sel({"pctl": 0.5}).plot(ax=ax, hue="gcm_id", color=exp["color"], ls=exp["ls"], label=exp["title"], lw=1)
            
        ax.set_ylabel("Contribution to sea-level (mm))")
        ax.set_xlabel(None)
        ax.set_title(None)
        ax.axhline(y=0, ls="dotted", lw=0.5)
        # # Create a legend outside the figure at the bottom middle
        handles, labels = ax.get_legend_handles_labels()
        legend_main = fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.1, 0.9), ncol=1)
        legend_main.set_title(None)
        legend_main.get_frame().set_linewidth(0.0)
        legend_main.get_frame().set_alpha(0.0)
        fig.tight_layout()
        plt.show()
        fig.savefig(f"pism_kitp_{basin_name.item()}_{res}.png", dpi=300)
        fig.savefig(f"pism_kitp_{basin_name.item()}_{res}.pdf")
        plt.close()
        del fig
    

In [ ]:
%%time
spatial_lowres_files = list(Path("/media/andy/LHS800DATA/2026_03_kitp_1500m/output/spatial/").glob("clipped_*.nc"))

def load_dataset(filename_or_obj: str | Path) -> xr.Dataset:
    with ProgressBar():
        ds = xr.open_mfdataset(filename_or_obj,
                                      preprocess=partial(preprocess, 
                                                         exp_regexp=r"id_(.+?)_uq_",                                                                                                                     
                                                         uq_regexp="uq_(.+?)_",   
                                                         uq_dim="uq_id",
                                                         exp_dim="exp_id"), 
                                      engine="netcdf4",
                                      parallel=True,
                                      chunks="auto",
                                      data_vars="minimal",
                                      join="outer",
                                      decode_times=time_coder,
                                      decode_timedelta=delta_coder).pint.quantify()
    ds["exp_id"] = ds["exp_id"].str.replace("CESM1-WACCM-SC_", "") 
 
    return ds

with dask.config.set(**{"array.slicing.split_large_chunks": False}):
    spatial_lowres = load_dataset(spatial_lowres_files).sel({"exp_id": exps})

In [ ]:
speed = spatial_lowres.isel({"time": [0, -1]})["velsurf_mag"].quantile(0.50, dim="uq_id", skipna=True)

In [ ]:
fg = spatial.climatic_mass_balance.resample(time="YE").mean(dim="time").plot(row="eb_id", col="exp_id", vmin=-2500, vmax=2500, cmap="RdBu")
fg.fig.savefig("smb.png", dpi=300)

In [ ]:
import cartopy.crs as ccrs 
proj = ccrs.Stereographic(central_latitude=90, central_longitude=-45, true_scale_latitude=70, globe=None)

In [ ]:
import math
import cartopy.crs as ccrs 
proj = ccrs.Stereographic(central_latitude=90, central_longitude=-45, true_scale_latitude=70, globe=None)

cmb = spatial.sel(eb_id="debm").climatic_mass_balance.resample(time="YE").mean(dim="time")
ice_density = xr.DataArray(910).pint.quantify("kg m^-3")
cmb = cmb.pint.quantify() / ice_density
cmb_anomalies = cmb - cmb.sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019").squeeze()
cmb_anomalies = cmb_anomalies.drop_sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019") 

dx = float(cmb.x.max() - cmb.x.min())                                                                                                                                              
dy = float(cmb.y.max() - cmb.y.min())                     
aspect = dy / dx                                                                                                                                                                   
                                                                                                                                                                                     
width = 6.4                                                                                                                                                                        

with mpl.rc_context(rc=rc_params):
    height = 2.6
    fg = cmb.plot(col="exp_id", 
                  vmin=-2.5,
                  vmax=2.5, 
                  cmap=cm_rdbu,
                  figsize=(width, height),             
                  cbar_kwargs={"shrink": 0.60,},                                                                                                                    
                  subplot_kws={"projection": proj},                                                                                                                              
                  transform=proj,                                                                   
                 )
    for ax in fg.axs.flat:
        ax.coastlines(linewidth=0.5)
        ax.set_extent([cmb.x.min(), cmb.x.max(), cmb.y.min(), cmb.y.max()], crs=proj)
    fig = fg.fig
    fig.savefig("smb.png", dpi=300)
    del fig

    height = 2.4
    fg = cmb_anomalies.plot(col="exp_id", 
                  vmin=-1,
                  vmax=1, 
                  cmap=cm_rdbu,
                  figsize=(width, height),             
                  cbar_kwargs={"shrink": 0.60,},                                                                                                                    
                  subplot_kws={"projection": proj},                                                                                                                              
                  transform=proj,                                                                   
                 )
    for ax in fg.axs.flat:
        ax.coastlines(linewidth=0.5)
        ax.set_extent([cmb.x.min(), cmb.x.max(), cmb.y.min(), cmb.y.max()], crs=proj)
    fig = fg.fig
    fg.fig.savefig("smb_anomalies.png", dpi=300)
    del fig

In [ ]:
import math
import cartopy.crs as ccrs 
proj = ccrs.Stereographic(central_latitude=90, central_longitude=-45, true_scale_latitude=70, globe=None)

seasons = {"djf": [0, 1, -1], "jja": [5, 6, 7]}

for season, time_steps in seasons.items():
    pr = spatial.sel(eb_id="debm").effective_precipitation.isel(time=time_steps).resample(time="YE").mean(dim="time")
    water_density = xr.DataArray(910).pint.quantify("kg m^-3")
    pr = pr.pint.quantify() / water_density
    pr  = pr.pint.to("mm/day")
    
    pr_anomalies = pr - cmb.sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019").squeeze()
    pr_anomalies = pr_anomalies.drop_sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019") 
    
    dx = float(cmb.x.max() - cmb.x.min())                                                                                                                                              
    dy = float(cmb.y.max() - cmb.y.min())                     
    aspect = dy / dx                                                                                                                                                                   
                                                                                                                                                                                         
    width = 6.4                                                                                                                                                                        
    
    with mpl.rc_context(rc=rc_params):
    
        height = 2.4
        fg = pr_anomalies.plot(col="exp_id", 
                      vmin=-1.5,
                      vmax=1.5, 
                      cmap="PiYG",
                      figsize=(width, height),             
                      cbar_kwargs={"shrink": 0.95,},                                                                                                                    
                      subplot_kws={"projection": proj},                                                                                                                              
                      transform=proj,                                                                   
                     )
        for ax in fg.axs.flat:
            ax.coastlines(linewidth=0.5)
            ax.set_extent([pr.x.min(), pr.x.max(), pr.y.min(), pr.y.max()], crs=proj)
        fig = fg.fig
        fg.fig.savefig(f"precip_anomalies_{season}.png", dpi=300)
        del fig

In [ ]:
import math
import cartopy.crs as ccrs 
proj = ccrs.Stereographic(central_latitude=90, central_longitude=-45, true_scale_latitude=70, globe=None)

for season, time_steps in seasons.items():
    tas = spatial.sel(eb_id="debm").effective_air_temp.resample(time="YE").mean(dim="time")
    tas_anomalies = tas - tas.sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019").squeeze()
    tas_anomalies = tas_anomalies.drop_sel(exp_id="HIRHAM5-ERA5_YMM_1990_2019") 
    
    dx = float(tas.x.max() - tas.x.min())                                                                                                                                              
    dy = float(tas.y.max() - tas.y.min())                     
    aspect = dy / dx                                                                                                                                                                   
                                                                                                                                                                                         
    width = 6.4                                                                                                                                                                        
    
    with mpl.rc_context(rc=rc_params):    
        height = 2.4
        fg = tas_anomalies.plot(col="exp_id", 
                      vmin=-3,
                      vmax=3, 
                      cmap="RdBu_r",
                      figsize=(width, height),             
                      cbar_kwargs={"shrink": 0.95,},                                                                                                                    
                      subplot_kws={"projection": proj},                                                                                                                              
                      transform=proj,                                                                   
                     )
        for ax in fg.axs.flat:
            ax.coastlines(linewidth=0.5)
            ax.set_extent([cmb.x.min(), cmb.x.max(), cmb.y.min(), cmb.y.max()], crs=proj)
        fig = fg.fig
        fg.fig.savefig(f"tas_anomalies_{season}.png", dpi=300)
        del fig

In [ ]:
obs = xr.open_dataset("/Volumes/LHS800/kitp_greenland/input/v3/HIRHAM5-ERA5_YMM_1990_2019_v3.nc")

In [ ]:
obs_ym = obs.resample(time="YE").mean(dim="time")

In [ ]:
%%time
run ~/base/pism-terra/pism_terra/kitp/analyze.py /Volumes/LHS800DATA/2026_04_kitp_v4/output/scalar/scalar_g2400m_*.nc

In [ ]:
import rioxarray
from pyproj import CRS                                                                                                                                                             
import geopandas as gpd

# Build the rotated pole CRS                                                                                                                                                       
rot_crs = CRS.from_cf({
      "grid_mapping_name": "rotated_latitude_longitude",                                                                                                                             
      "grid_north_pole_longitude": 160.0,                                                                                                                                            
      "grid_north_pole_latitude": 18.0,
      "north_pole_grid_longitude": 0.0,                                                                                                                                              
})                                                        
                                                                                                                                                                                     
basin = gpd.read_file("/Users/andy/base/pism-ragis/data/mouginot/Greenland_Basins_PS_v1.4.2_w_shelves.gpkg")  
column = "SUBREGION1"

hh5 = xr.open_dataset("/Users/andy/base/pism-ragis/data/climate/HH5_MEAN.nc").squeeze().rio.write_crs(rot_crs).rio.set_spatial_dims(x_dim="rlon", y_dim="rlat")                                                                                                                 
mar = xr.open_dataset("/Users/andy/base/pism-ragis/data/climate/MARv3.14_MEAN.nc", decode_times=False, decode_timedelta=False).squeeze().rio.write_crs("EPSG:3413").rio.set_spatial_dims(x_dim="x", y_dim="y").drop_vars("time_bnds")                                                                                                                    
racmo = xr.open_dataset("/Users/andy/base/pism-ragis/data/climate/RACMO2.3p2_MEAN.nc", decode_times=False, decode_timedelta=False).squeeze().rio.write_crs(rot_crs).rio.set_spatial_dims(x_dim="rlon", y_dim="rlat")                                                                                                               

hh5_m = hh5.rio.reproject_match(spatial.rio.write_crs("EPSG:3413")).rio.clip(basin[basin[column] == "GIS"].geometry, drop=False)
#mar_m = mar.rio.reproject_match(spatial.rio.write_crs("EPSG:3413")).rio.clip(basin[basin[column] == "GIS"].geometry, drop=False).pint.quantify()
racmo_m = racmo.rio.reproject_match(spatial.rio.write_crs("EPSG:3413")).rio.clip(basin[basin[column] == "GIS"].geometry, drop=False)

hh5_m["x"].attrs.pop("units", None)                                                                                                                                                
hh5_m["y"].attrs.pop("units", None) 

racmo_m["x"].attrs.pop("units", None)                                                                                                                                                
racmo_m["y"].attrs.pop("units", None) 

hh5_cmb = (hh5_m.climatic_mass_balance.pint.quantify() / ice_density).pint.to("m/yr")
racmo_cmb = (racmo_m.climatic_mass_balance.pint.quantify() / ice_density).pint.to("m/yr")



In [ ]:
with mpl.rc_context(rc=rc_params):
    fig, axs = plt.subplots(1, 2, figsize=(6.4, 2.8))
    hh5_cmb.plot(    
        vmin=-2.5,
        vmax=2.5, 
        cmap=cm_rdbu,
        ax=axs[0],
        add_colorbar=False
    )
    # racmo_cmb.plot(    
    #     vmin=-2.5,
    #     vmax=2.5, 
    #     cmap=cm_rdbu,
    #     ax=axs[1],
    #     add_colorbar=False
    # )
    fig.savefig("rcm_smb.png", dpi=300)

In [ ]:
        ice_mass = (scalar_lowres["ice_mass"].pint.to("Gt") * gt2mmsle).pint.dequantify()
        #ice_mass = ice_mass.isel({"time": 0})


In [ ]:
baseline_opts

In [ ]:
cmb.sum(dim=["x", "y"]).values

In [ ]:
fig, ax =  plt.subplots(1,1)
cmb = spatial.sel(eb_id="debm").climatic_mass_balance.sum(dim=["x", "y"])
cmb.plot(hue="exp_id", ax=ax)
plt.show()

In [ ]:
cmb

In [ ]:
_grounding_line_flux.values

In [ ]:
_ds.grounding_line_flux.values

In [ ]:
ice_mass.sel({"exp_id": "HIRHAM5-ERA5_YMM_1990_2019"}).squeeze()

In [ ]:
ice_mass.values

In [ ]:
scalar = load_dataset(scalar_files,
                                 exp_regexp=r"id_(.+?)_0",                                                                                                                     
                                 uq_regexp=None,   
                                 uq_dim=None,
                                 exp_dim="exp_id",
                                 process_config=True)

In [ ]:
scalar.pism_config

In [ ]:
preprocess?

In [ ]:
  import netCDF4                                                                                                                                                                     
  nc = netCDF4.Dataset(scalar_files[0])
  print("pism_config" in nc.variables)
  nc.close() 

In [ ]:
scalar_files

In [ ]:
ds = xr.open_dataset(scalar_files[0])

In [ ]:
pds = preprocess(ds,                                 
           exp_regexp=r"id_(.+?)_0",                                                                                                                     
           uq_regexp=None,   
           uq_dim=None,
           exp_dim="exp_id",
          )

In [ ]:
scalar.time

In [ ]:
scalar

In [ ]:
exp_norm

In [ ]:
del exps_norm["HIRHAM5-ERA5_YMM_1990_2019"]

In [ ]:
scalar_norm

In [ ]:
baseline.pism_config

In [ ]:
experiments.ice_area_glacierized.values

In [ ]:
slc